# NB12 — Diagnostics before any redesign: C2 (feature stability) + F3 (error overlap)

Two cheap analyses that tell us **why** the hybrid lost on deepseek and won on gpt, before we spend a
full retrain cycle on any architectural change.

- **C2 · Feature stability across generators** (CPU, seconds). For each of the NEW_5 features, how much
  does its distribution SHIFT from one generator to another? A feature whose mean moves a lot between
  generators is a feature whose learned weight becomes actively misleading when that generator is
  held out. Prime suspect for the deepseek loss.
- **F3 · Error-overlap analysis** (GPU, ~20-30 min). Load the 12 saved NB11 checkpoints, re-run
  inference only (no training), and split every test article into four buckets: both right, both wrong,
  **hybrid fixed it** (neural wrong → hybrid right), **hybrid broke it** (neural right → hybrid wrong).
  This turns "+0.82 on gpt / −0.70 on deepseek" into a list of actual articles we can inspect.

Single top-to-bottom pass. Nothing is retrained — checkpoints are loaded read-only.

## 1 · Config

In [1]:
import os, json, numpy as np, pandas as pd, torch
P_DATASET = "/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet"          # EDIT
P_VSTAT16 = "/kaggle/input/notebooks/bahaaqassem/ph2-nb6e-extract-11-features/vstat16_scaled.parquet"   # EDIT
# NB11 outputs promoted to a dataset (EDIT to the exact Input-pane paths):
# NB11 ran in TWO commits, and /kaggle/working starts clean each time -> the 12 checkpoints are
# split across TWO datasets. List every folder that may hold them; all are searched.
P_CKPT_DIRS = [
    "/kaggle/input/datasets/bahaaqassem/trackb-logo-all-generators/logo_ckpt",
    "/kaggle/input/notebooks/bahaaqassem/nb11-trackb-logo-all-generators/logo_ckpt",
]
P_CHUNKS  = "/kaggle/input/notebooks/bahaaqassem/nb11-trackb-logo-all-generators/chunks_K9.npz"
P_RESULTS = "/kaggle/input/notebooks/bahaaqassem/nb11-trackb-logo-all-generators/logo_results.json"

MODEL_ID   = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
STAT_COLS  = ["burstiness","ttr","quote_ratio","function_word_ratio","compressibility"]
GENERATORS = ["deepseek","sonnet","qwen","gemini","gpt","opus"]
K_CHUNKS, MAX_CT = 9, 510
HIDDEN, DROPOUT = 256, 0.0
DEV = "cuda" if torch.cuda.is_available() else "cpu"
RUN_F3 = True     # set False to run only the CPU part (C2)
print("config loaded | device", DEV)

config loaded | device cuda


In [2]:
# what is actually on disk? (fix paths from this listing if anything is missing)
import glob
found = {}
for d in P_CKPT_DIRS:
    if not os.path.isdir(d):
        print(f"  [dir not found] {d}"); continue
    fs = sorted(os.path.basename(p) for p in glob.glob(os.path.join(d, "*.pt")))
    print(f"  [{len(fs):2d} files] {d}")
    for f in fs: found[f] = d
need = [f"{g}_{m}.pt" for g in GENERATORS for m in ("hybrid","neural")]
missing = [f for f in need if f not in found]
print(f"\nfound {len(need)-len(missing)}/12 checkpoints")
if missing: print("MISSING:", ", ".join(missing))

  [ 7 files] /kaggle/input/datasets/bahaaqassem/trackb-logo-all-generators/logo_ckpt
  [ 5 files] /kaggle/input/notebooks/bahaaqassem/nb11-trackb-logo-all-generators/logo_ckpt

found 12/12 checkpoints


## 2 · Load data

In [3]:
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
v16 = pd.read_parquet(P_VSTAT16)
if "article_id" in v16.columns: v16 = v16.set_index("article_id")
df = df.loc[df.index.intersection(v16.index)]; v16 = v16.loc[df.index]
assert (df.index == v16.index).all()

Xstat = v16[STAT_COLS].to_numpy(np.float32)
y     = df["label"].to_numpy(np.int64)
gen   = df["generator"].fillna("__human__").to_numpy()
split = df["split"].to_numpy()
print(f"{len(df)} articles | AI {int((y==1).sum())} | human {int((y==0).sum())}")

7101 articles | AI 3601 | human 3500


# Part One — C2 · Feature stability across generators

The reasoning: the model learns feature weights from **five seen generators**. If the mean of a feature
on the held-out generator lies far from its mean on those five, the learned weight becomes
**actively misleading** rather than neutral.

We measure this directly: for each (feature, held-out generator) pair we compute the **shift** =
|mean of the feature on the held-out generator − its mean on the remaining five| divided by the
within-generator standard deviation (a standardized effect size, Cohen's d).

## 3 · Mean of each feature per generator

In [4]:
ai = (y == 1)
rows = []
for g in GENERATORS:
    m = ai & (gen == g)
    rows.append({"generator": g, "n": int(m.sum()),
                 **{f: round(float(Xstat[m, i].mean()), 3) for i, f in enumerate(STAT_COLS)}})
rows.append({"generator": "HUMAN", "n": int((~ai).sum()),
             **{f: round(float(Xstat[~ai, i].mean()), 3) for i, f in enumerate(STAT_COLS)}})
means = pd.DataFrame(rows)
import IPython.display as ipd
print("Mean of each feature (scaled) per generator:"); ipd.display(means); print(means.to_string(index=False))

متوسط كل خاصية (scaled) لكل مولّد:


,generator,n,burstiness,ttr,quote_ratio,function_word_ratio,compressibility
0,deepseek,878,-0.451,-0.080,0.152,0.535,-0.341
1,sonnet,840,-0.725,-0.262,-0.209,0.829,-0.050
2,qwen,622,-0.246,1.277,-0.676,-0.186,0.718
3,gemini,441,-0.568,0.858,-0.742,-0.011,0.245
4,gpt,440,-0.240,0.594,-0.014,-0.197,0.290
5,opus,380,-0.556,0.016,-0.599,0.736,-0.120
6,HUMAN,3500,0.494,-0.347,0.299,-0.356,-0.086


generator    n  burstiness    ttr  quote_ratio  function_word_ratio  compressibility
 deepseek  878      -0.451 -0.080        0.152                0.535           -0.341
   sonnet  840      -0.725 -0.262       -0.209                0.829           -0.050
     qwen  622      -0.246  1.277       -0.676               -0.186            0.718
   gemini  441      -0.568  0.858       -0.742               -0.011            0.245
      gpt  440      -0.240  0.594       -0.014               -0.197            0.290
     opus  380      -0.556  0.016       -0.599                0.736           -0.120
    HUMAN 3500       0.494 -0.347        0.299               -0.356           -0.086


## 4 · Shift of each feature when each generator is held out (LOGO shift)

In [5]:
shift = []
for g in GENERATORS:
    held = ai & (gen == g)
    seen = ai & (gen != g)
    row = {"held_out": g}
    for i, f in enumerate(STAT_COLS):
        mu_h, mu_s = Xstat[held, i].mean(), Xstat[seen, i].mean()
        sd_pool = np.sqrt(0.5*(Xstat[held, i].var() + Xstat[seen, i].var())) + 1e-9
        row[f] = round(float(abs(mu_h - mu_s)/sd_pool), 3)     # standardized shift (|d|)
    shift.append(row)
S = pd.DataFrame(shift)
print("Standardized shift |d| per feature when the generator is held out (larger = riskier):")
ipd.display(S); print(S.to_string(index=False))

print("\nMean shift per feature across all generators (instability measure):")
inst = S[STAT_COLS].mean().sort_values(ascending=False)
for f, v in inst.items():
    print(f"  {f:24s} {v:.3f}")
print("\nHighest shift = least stable = the candidate for breaking generalization.")

الإزاحة المعيارية |d| للخاصية عند حجب المولّد (أكبر = أخطر):


,held_out,burstiness,ttr,quote_ratio,function_word_ratio,compressibility
0,deepseek,0.052,0.649,0.875,0.305,0.526
1,sonnet,0.449,1.001,0.170,0.782,0.169
2,qwen,0.252,1.619,0.947,0.711,0.769
3,gemini,0.137,0.845,1.143,0.505,0.187
4,gpt,0.397,0.398,0.517,0.758,0.237
5,opus,0.127,0.449,0.718,0.547,0.230


held_out  burstiness   ttr  quote_ratio  function_word_ratio  compressibility
deepseek       0.052 0.649        0.875                0.305            0.526
  sonnet       0.449 1.001        0.170                0.782            0.169
    qwen       0.252 1.619        0.947                0.711            0.769
  gemini       0.137 0.845        1.143                0.505            0.187
     gpt       0.397 0.398        0.517                0.758            0.237
    opus       0.127 0.449        0.718                0.547            0.230

متوسط الإزاحة لكل خاصية عبر كل المولّدات (مقياس عدم الاستقرار):
  ttr                      0.827
  quote_ratio              0.728
  function_word_ratio      0.601
  compressibility          0.353
  burstiness               0.236

الأعلى إزاحة = الأقل استقراراً = المرشّح لإفساد التعميم.


## 5 · Relating the shift to the actual per-fold result

In [6]:
LIFT = {"deepseek": -0.70, "sonnet": +0.23, "qwen": 0.00,
        "gemini": 0.00, "gpt": +0.82, "opus": 0.00}      # NB11 (hybrid − neural)
S2 = S.copy()
S2["max_shift"]  = S2[STAT_COLS].max(axis=1)
S2["worst_feat"] = S2[STAT_COLS].idxmax(axis=1)
S2["mean_shift"] = S2[STAT_COLS].mean(axis=1)
S2["fusion_lift"] = S2["held_out"].map(LIFT)
_t = S2[["held_out","fusion_lift","mean_shift","max_shift","worst_feat"]]
ipd.display(_t); print(_t.to_string(index=False))

r = np.corrcoef(S2["mean_shift"], S2["fusion_lift"])[0,1]
print(f"\ncorrelation (mean shift <-> fusion lift) = {r:+.3f}")
print("A strong negative correlation implies: the more the features shift on the held-out generator, the less fusion helps (or the more it hurts).")
print("Note: only 6 points — a directional signal, not statistical evidence.")

,held_out,fusion_lift,mean_shift,max_shift,worst_feat
0,deepseek,-0.70,0.4814,0.875,quote_ratio
1,sonnet,0.23,0.5142,1.001,ttr
2,qwen,0.00,0.8596,1.619,ttr
3,gemini,0.00,0.5634,1.143,quote_ratio
4,gpt,0.82,0.4614,0.758,function_word_ratio
5,opus,0.00,0.4142,0.718,quote_ratio


held_out  fusion_lift  mean_shift  max_shift          worst_feat
deepseek        -0.70      0.4814      0.875         quote_ratio
  sonnet         0.23      0.5142      1.001                 ttr
    qwen         0.00      0.8596      1.619                 ttr
  gemini         0.00      0.5634      1.143         quote_ratio
     gpt         0.82      0.4614      0.758 function_word_ratio
    opus         0.00      0.4142      0.718         quote_ratio

ارتباط (متوسط الإزاحة ↔ مكسب الاندماج) = -0.083
ارتباط سالب قوي ⟹ كلّما زاد انزياح الخصائص على المولّد المحجوب، قلّ نفع الاندماج (أو ضرّ).
ملاحظة: 6 نقاط فقط — إشارة اتجاهية لا دليلاً إحصائياً.


# Part Two — F3 · Error-overlap analysis

We load the saved NB11 checkpoints (**with no training at all**) and extract one prediction per article,
then split each fold's articles into four buckets. Two of them matter: **hybrid fixed it** and
**hybrid broke it** — these are what explain the +0.82 and the −0.70.

## 6 · Rebuilding the model and the data loader (identical to NB11)

In [7]:
if RUN_F3:
    !pip install -q transformers
    import torch.nn as nn
    from transformers import AutoModel, AutoTokenizer
    from torch.utils.data import DataLoader, Dataset

    tok = AutoTokenizer.from_pretrained(MODEL_ID); PAD = tok.pad_token_id
    z = np.load(P_CHUNKS, allow_pickle=True); CH, NCH = z["ch"], z["nch"]
    assert CH.shape[0] == len(df), f"chunks {CH.shape[0]} != articles {len(df)}"
    print("chunks:", CH.shape)

    class Net(nn.Module):
        def __init__(self, stat_dim, use_stat):
            super().__init__()
            self.use_stat = use_stat
            self.enc = AutoModel.from_pretrained(MODEL_ID)
            h = self.enc.config.hidden_size; fused = h + (stat_dim if use_stat else 0)
            self.register_buffer("mu", torch.zeros(fused)); self.register_buffer("sd", torch.ones(fused))
            self.head = nn.Sequential(nn.Linear(fused, HIDDEN), nn.LayerNorm(HIDDEN), nn.ReLU(),
                                      nn.Dropout(DROPOUT), nn.Linear(HIDDEN, 2))
        def encode(self, ids, nch):
            B,K,L = ids.shape; flat = ids.view(B*K, L); att = (flat != PAD).long()
            cls = self.enc(input_ids=flat, attention_mask=att).last_hidden_state[:,0,:].view(B,K,-1)
            m = (torch.arange(K, device=ids.device)[None,:] < nch[:,None]).float().unsqueeze(-1)
            return (cls*m).sum(1)/m.sum(1).clamp(min=1)
        def forward(self, ids, nch, stat):
            v = self.encode(ids, nch)
            if self.use_stat: v = torch.cat([v, stat], 1)
            v = (v - self.mu)/self.sd
            return self.head(v)

    class DS(Dataset):
        def __init__(self, rows): self.rows = rows
        def __len__(self): return len(self.rows)
        def __getitem__(self, k):
            i = self.rows[k]
            return (torch.from_numpy(CH[i].astype(np.int64)), int(NCH[i]),
                    torch.from_numpy(Xstat[i]), int(y[i]), i)
    def collate(b):
        return (torch.stack([x[0] for x in b]), torch.tensor([x[1] for x in b]),
                torch.stack([x[2] for x in b]), torch.tensor([x[3] for x in b]),
                [x[4] for x in b])
    print("model + loader ready")

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

chunks: (7101, 9, 512)
model + loader ready


## 7 · Extracting predictions from the 12 checkpoints (inference only)

In [8]:
if RUN_F3:
    @torch.no_grad()
    def predict(model, rows):
        model.eval(); P=[]; I=[]
        for ids,nch,st,yy,idx in DataLoader(DS(rows), batch_size=8, collate_fn=collate):
            ids,nch,st = ids.to(DEV), nch.to(DEV), st.to(DEV)
            with torch.amp.autocast('cuda'):
                p = torch.softmax(model(ids,nch,st), 1)[:,1]
            P.append(p.float().cpu().numpy()); I += idx
        return np.concatenate(P), np.array(I)

    is_test = (split == "test"); human = (y == 0)
    preds = {}
    for g in GENERATORS:
        te = np.where(is_test & (human | (gen == g)))[0].tolist()
        for use_stat, name in [(True,"hybrid"), (False,"neural")]:
            ck = None
            for d in P_CKPT_DIRS:
                p = os.path.join(d, f"{g}_{name}.pt")
                if os.path.exists(p): ck = p; break
            if ck is None:
                print(f"  !! missing {g}_{name}.pt in all P_CKPT_DIRS"); continue
            m = Net(len(STAT_COLS), use_stat).to(DEV)
            # weights_only=False: PyTorch>=2.6 defaults to True and refuses our ckpt (it stores numpy RNG state).
            # These files are our own output, so loading them fully is safe.
            m.load_state_dict(torch.load(ck, map_location=DEV, weights_only=False)["model"])
            # sanity: the standardizer buffers must not be at their init values
            if float(m.mu.abs().sum()) == 0.0:
                print(f"  !! WARNING {g}:{name} has an unfitted standardizer")
            p, idx = predict(m, te)
            preds[f"{g}:{name}"] = (p, idx)
            del m; torch.cuda.empty_cache()
            print(f"  got {g}:{name}  ({len(te)} articles)")
    print("done")

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

  got deepseek:hybrid  (674 articles)


...

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  got opus:hybrid  (602 articles)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  got opus:neural  (602 articles)
done


## 8 · Error-overlap buckets per fold

In [9]:
if RUN_F3:
    from sklearn.metrics import f1_score
    summary = []; detail = {}
    for g in GENERATORS:
        kh, kn = f"{g}:hybrid", f"{g}:neural"
        if kh not in preds or kn not in preds: continue
        ph, ih = preds[kh]; pn, iN = preds[kn]
        assert (ih == iN).all(), "index mismatch"
        yt = y[ih]
        hc = ((ph >= .5).astype(int) == yt)      # hybrid correct
        nc = ((pn >= .5).astype(int) == yt)      # neural correct
        fixed  = np.where(hc & ~nc)[0]
        broke  = np.where(~hc & nc)[0]
        summary.append({"held_out": g, "n": len(yt),
                        "both_right": int((hc & nc).sum()), "both_wrong": int((~hc & ~nc).sum()),
                        "hybrid_FIXED": len(fixed), "hybrid_BROKE": len(broke),
                        "net": len(fixed) - len(broke),
                        "F1_hybrid": round(100*f1_score(yt,(ph>=.5).astype(int),average="macro"),2),
                        "F1_neural": round(100*f1_score(yt,(pn>=.5).astype(int),average="macro"),2)})
        detail[g] = {"idx": ih, "yt": yt, "ph": ph, "pn": pn, "fixed": fixed, "broke": broke}
    SUM = pd.DataFrame(summary); ipd.display(SUM); print(SUM.to_string(index=False))
    print("hybrid_FIXED = fixed by the hybrid | hybrid_BROKE = broken by it | net = net effect in number of articles")

,held_out,n,both_right,both_wrong,hybrid_FIXED,hybrid_BROKE,net,F1_hybrid,F1_neural
0,deepseek,674,669,2,0,3,-3,98.83,99.53
1,sonnet,676,674,1,1,0,1,99.77,99.54
2,qwen,627,625,2,0,0,0,99.34,99.34
3,gemini,602,601,1,0,0,0,99.56,99.56
4,gpt,607,603,2,2,0,2,99.18,98.36
5,opus,602,601,1,0,0,0,99.56,99.56


held_out   n  both_right  both_wrong  hybrid_FIXED  hybrid_BROKE  net  F1_hybrid  F1_neural
deepseek 674         669           2             0             3   -3      98.83      99.53
  sonnet 676         674           1             1             0    1      99.77      99.54
    qwen 627         625           2             0             0    0      99.34      99.34
  gemini 602         601           1             0             0    0      99.56      99.56
     gpt 607         603           2             2             0    2      99.18      98.36
    opus 602         601           1             0             0    0      99.56      99.56
hybrid_FIXED = أصلحها الهجين | hybrid_BROKE = أفسدها | net = صافي الأثر بعدد المقالات


## 9 · Inspecting the affected articles (gpt and deepseek)

In [10]:
if RUN_F3:
    for g in ["gpt", "deepseek"]:
        if g not in detail: continue
        d = detail[g]
        print("="*66); print(f"fold: {g}")
        for tag, sel in [("FIXED by hybrid", d["fixed"]), ("BROKEN by hybrid", d["broke"])]:
            if len(sel) == 0: print(f"  {tag}: none"); continue
            rows = []
            for k in sel:
                i = d["idx"][k]
                rows.append({"article_id": df.index[i], "true": int(d["yt"][k]),
                             "gen": gen[i], "p_neural": round(float(d["pn"][k]),3),
                             "p_hybrid": round(float(d["ph"][k]),3),
                             **{f: round(float(Xstat[i,j]),2) for j,f in enumerate(STAT_COLS)}})
            print(f"\n  {tag}  (n={len(sel)})"); _d = pd.DataFrame(rows); ipd.display(_d); print(_d.to_string(index=False))
    print("\nRead the feature columns: which feature was extreme in the fixed articles versus the broken ones?")

fold: gpt

  FIXED by hybrid  (n=2)


,article_id,true,gen,p_neural,p_hybrid,burstiness,ttr,quote_ratio,function_word_ratio,compressibility
0,AI_gpt_HA_00417,1,gpt,0.074,0.767,0.04,-0.83,-0.76,-1.20,0.94
1,HU_HA_02675,0,human,0.504,0.005,0.46,-0.29,-0.67,0.62,-0.18


     article_id  true   gen  p_neural  p_hybrid  burstiness   ttr  quote_ratio  function_word_ratio  compressibility
AI_gpt_HA_00417     1   gpt     0.074     0.767        0.04 -0.83        -0.76                -1.20             0.94
    HU_HA_02675     0 human     0.504     0.005        0.46 -0.29        -0.67                 0.62            -0.18
  BROKEN by hybrid: none
fold: deepseek
  FIXED by hybrid: none

  BROKEN by hybrid  (n=3)


,article_id,true,gen,p_neural,p_hybrid,burstiness,ttr,quote_ratio,function_word_ratio,compressibility
0,HU_HA_00654,0,human,0.434,0.977,-0.93,0.51,0.20,-0.51,0.79
1,AI_deepseek_HA_00592,1,deepseek,0.829,0.209,0.63,-0.52,-0.60,0.19,-0.35
2,AI_deepseek_HA_02719,1,deepseek,0.923,0.209,0.29,-0.50,1.51,0.53,-0.18


          article_id  true      gen  p_neural  p_hybrid  burstiness   ttr  quote_ratio  function_word_ratio  compressibility
         HU_HA_00654     0    human     0.434     0.977       -0.93  0.51         0.20                -0.51             0.79
AI_deepseek_HA_00592     1 deepseek     0.829     0.209        0.63 -0.52        -0.60                 0.19            -0.35
AI_deepseek_HA_02719     1 deepseek     0.923     0.209        0.29 -0.50         1.51                 0.53            -0.18

اقرأ عمود الخصائص: أي خاصية كانت متطرّفة في المقالات التي أُصلحت مقابل التي أُفسدت؟


## 10 · Which feature drove the decision in each case?

In [11]:
if RUN_F3:
    out = []
    for g in GENERATORS:
        if g not in detail: continue
        d = detail[g]
        for tag, sel in [("fixed", d["fixed"]), ("broke", d["broke"])]:
            if len(sel) == 0: continue
            ids_ = d["idx"][sel]
            base = Xstat[[i for i in range(len(df)) if split[i]=="test"]].mean(0)
            row = {"held_out": g, "bucket": tag, "n": len(sel)}
            for j,f in enumerate(STAT_COLS):
                row[f] = round(float(Xstat[ids_, j].mean() - base[j]), 3)   # deviation from the test-split mean
            out.append(row)
    D = pd.DataFrame(out); ipd.display(D); print(D.to_string(index=False))
    print("Values = deviation of the feature mean in this bucket from the test-split mean.")
    print("A feature extreme in fixed and moderate in broke implies it is the source of the gain; the reverse implies it is the source of the harm.")

,held_out,bucket,n,burstiness,ttr,quote_ratio,function_word_ratio,compressibility
0,deepseek,broke,3,-0.032,-0.095,0.350,0.079,0.117
1,sonnet,fixed,1,-0.470,-0.289,0.872,-1.921,0.239
2,gpt,fixed,2,0.219,-0.488,-0.737,-0.280,0.409


held_out bucket  n  burstiness    ttr  quote_ratio  function_word_ratio  compressibility
deepseek  broke  3      -0.032 -0.095        0.350                0.079            0.117
  sonnet  fixed  1      -0.470 -0.289        0.872               -1.921            0.239
     gpt  fixed  2       0.219 -0.488       -0.737               -0.280            0.409
القيم = انحراف متوسط الخاصية في هذه الفئة عن متوسط مجموعة الاختبار.
خاصية متطرّفة في fixed ومعتدلة في broke ⟹ هي مصدر المكسب، والعكس مصدر الضرر.
